![DB Academy](https://files.training.databricks.com/binder/prod_main/get-started-with-databricks-for-data-warehousing-en_us-3.0.3/images/20261012T131742Z/Get Started with Databricks for Data Warehousing/Includes/images/db-academy.png)

<div style="max-width: 1000px; margin: 0 auto; font-family: sans-serif;">
<div style="background: #1B5162; color: white; border-radius: 8px; padding: 28px 32px; text-align: center; position: relative;">
  <div style="font-size: 14pt; font-weight: 600; text-transform: uppercase; letter-spacing: 1px; opacity: 0.85; margin-bottom: 8px;">Lesson 10</div>
  <div style="font-size: 24pt; font-weight: 700; line-height: 1.3;">Deliver Insights with Metric Views</div>
  <div style="font-size: 14pt; margin-top: 12px; opacity: 0.9;">Read a metric view's YAML definition in the Catalog Explorer UI, then learn how to query it.</div>
</div>
</div>

## REQUIRED: SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select Serverless Compute</strong>
  <div style="color:#333;">

This notebook runs on Serverless compute. Before running any cells, confirm Serverless is selected in the compute dropdown at the top-right of this notebook. This course was developed and tested on <strong>Serverless environment version 5</strong>.
  </div>
</div>

### Setup
Run the cell below to configure your environment for this lesson. The setup script will initialize your catalog and schema, and **create the demo metric view** you will explore in this lesson. The metric view is created for you automatically so this lesson can focus on reading and querying it rather than authoring YAML.

In [0]:
%run ./Includes/Classroom-Setup-10

## What this lesson covers

You have built a clean star schema. Anyone with SQL skills can now answer business questions by writing the right `SELECT`. But not everyone in the business writes SQL, and even SQL writers benefit from a shared definition of what counts as "revenue" or "order count" so two teams do not end up with two different numbers for the same metric.

A **metric view** is Unity Catalog's answer to this problem. You define your dimensions and measures once, in one place, using a YAML specification authored in the Catalog Explorer UI. Every consumer (SQL Editor, AI/BI dashboards, Genie Agents, alerts, and external BI tools like Power BI and Tableau) then reads from the same definition.

In this lesson you will:

1. See why metric views matter for keeping metric definitions consistent.
2. Open a metric view that was pre-created for you in Catalog Explorer.
3. Explore it in the low-code UI (joins, measures, dimensions).
4. Walk through the YAML definition section by section.
5. Query the metric view, and use Genie Code to explain a query.

## 1. Why metric views exist

Without metric views, every consuming tool defines metrics on its own: the SQL analyst writes `SUM(total_amount)`, the dashboard author wires up a different aggregation, the Power BI report duplicates the logic, and a Genie Agent infers something plausible. Five tools, four definitions of "revenue," and a long argument in the next quarterly meeting.

With a metric view, all consumers read the same definition.

<div style="width: 100%; max-width: 880px; margin: 16px auto;">
<svg viewBox="0 0 920 520" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" style="width: 100%; height: auto; display: block;">
<text x="460" y="30" text-anchor="middle" fill="#1B5162" font-size="19" font-weight="700">ONE METRIC VIEW, MANY CONSUMERS</text>
<text x="460" y="54" text-anchor="middle" fill="#5E7077" font-size="15" font-style="italic">Define your business metrics once. Every tool reads the same definition.</text>
<!-- upstream tables feeding the metric view -->
<rect x="276" y="76" width="368" height="44" rx="6" fill="#FFFFFF" stroke="#1B5162" stroke-width="1.2" stroke-dasharray="5,3"/>
<text x="460" y="103" text-anchor="middle" fill="#5E7077" font-size="15">fact_orders + dimensions (the star schema)</text>
<line x1="460" y1="120" x2="460" y2="150" stroke="#9AA7AD" stroke-width="1.5"/>
<text x="476" y="139" text-anchor="start" fill="#5E7077" font-size="15" font-style="italic">loaded by Spark Declarative Pipelines / Auto Loader</text>
<!-- the metric view -->
<rect x="310" y="150" width="300" height="96" rx="10" fill="#F9F7F4" stroke="#FE3722" stroke-width="2.5"/>
<text x="460" y="180" text-anchor="middle" fill="#0b2026" font-size="17" font-weight="700">vw_product_metrics</text>
<text x="460" y="202" text-anchor="middle" fill="#FE3722" font-size="15" font-weight="700" letter-spacing="1">METRIC VIEW</text>
<text x="460" y="226" text-anchor="middle" fill="#5E7077" font-size="15">measures: total_revenue, order_count</text>
<!-- five consumer boxes -->
<rect x="20" y="330" width="165" height="120" rx="8" fill="#FFFFFF" stroke="#1B5162" stroke-width="1.5"/>
<text x="102" y="362" text-anchor="middle" fill="#0b2026" font-size="15" font-weight="700">SQL Editor</text>
<text x="102" y="386" text-anchor="middle" fill="#5E7077" font-size="15">ad-hoc analyst</text>
<text x="102" y="404" text-anchor="middle" fill="#5E7077" font-size="15">queries</text>
<rect x="200" y="330" width="165" height="120" rx="8" fill="#FFFFFF" stroke="#1B5162" stroke-width="1.5"/>
<text x="282" y="362" text-anchor="middle" fill="#0b2026" font-size="15" font-weight="700">AI/BI</text>
<text x="282" y="382" text-anchor="middle" fill="#0b2026" font-size="15" font-weight="700">Dashboards</text>
<text x="282" y="406" text-anchor="middle" fill="#5E7077" font-size="15">stakeholder views</text>
<rect x="380" y="330" width="165" height="120" rx="8" fill="#FFFFFF" stroke="#1B5162" stroke-width="1.5"/>
<text x="462" y="362" text-anchor="middle" fill="#0b2026" font-size="15" font-weight="700">Genie Agents</text>
<text x="462" y="386" text-anchor="middle" fill="#5E7077" font-size="15">natural-language</text>
<text x="462" y="404" text-anchor="middle" fill="#5E7077" font-size="15">questions</text>
<rect x="560" y="330" width="165" height="120" rx="8" fill="#FFFFFF" stroke="#1B5162" stroke-width="1.5"/>
<text x="642" y="362" text-anchor="middle" fill="#0b2026" font-size="15" font-weight="700">Alerts</text>
<text x="642" y="386" text-anchor="middle" fill="#5E7077" font-size="15">threshold-based</text>
<text x="642" y="404" text-anchor="middle" fill="#5E7077" font-size="15">notifications</text>
<rect x="740" y="330" width="165" height="120" rx="8" fill="#FFFFFF" stroke="#1B5162" stroke-width="1.5"/>
<text x="822" y="362" text-anchor="middle" fill="#0b2026" font-size="15" font-weight="700">External BI</text>
<text x="822" y="386" text-anchor="middle" fill="#5E7077" font-size="15">Power BI, Tableau</text>
<text x="822" y="404" text-anchor="middle" fill="#5E7077" font-size="15">via Partner Connect</text>
<!-- connectors metric view -> consumers -->
<line x1="380" y1="246" x2="102" y2="330" stroke="#618794" stroke-width="1.5"/>
<line x1="420" y1="246" x2="282" y2="330" stroke="#618794" stroke-width="1.5"/>
<line x1="460" y1="246" x2="462" y2="330" stroke="#618794" stroke-width="1.5"/>
<line x1="500" y1="246" x2="642" y2="330" stroke="#618794" stroke-width="1.5"/>
<line x1="540" y1="246" x2="822" y2="330" stroke="#618794" stroke-width="1.5"/>
<text x="460" y="486" text-anchor="middle" fill="#5E7077" font-size="15" font-style="italic">Each consumer evaluates the same measure definitions &#8212; no duplicated metric logic.</text>
</svg>
</div>

## 2. Open the metric view in Catalog Explorer

The setup at the top of this notebook created a metric view called **`vw_product_metrics`** in your `vintage_audio` schema. Open it now so you can read its YAML definition while you go through the rest of this lesson.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 14px 18px; border-radius: 4px; margin: 12px 0; font-size: 15px; color: #333; line-height: 1.6;">
  <strong style="color: #1565c0; font-size: 16px;">Try this in Catalog Explorer</strong>
  <ol style="margin: 8px 0 0 0; padding-left: 20px;">
    <li>In the left sidebar, right-click on <strong>Catalog</strong> and choose <strong>Open Link in New Tab</strong>. Keep this notebook open in its own tab.</li>
    <li>Navigate to your catalog (starting with <code style="font-size: 15px;">labuser</code>) &#x2192; <strong>vintage_audio</strong> schema.</li>
    <li>Click on <strong>vw_product_metrics</strong>. The metric view page has tabs: <strong>Overview</strong> (dimensions and measures), <strong>Details</strong> (YAML), <strong>Lineage</strong> (related assets), <strong>Permissions</strong>, and <strong>Insights</strong>.</li>
    <li>In the top-right of the page, click <strong>Edit</strong> to open the metric view editor.</li>
    <li>The editor has two modes (toggled top-right): <strong>UI</strong> (low-code visual editor) and <strong>YAML</strong> (raw definition). Start with <strong>UI</strong>.</li>
  </ol>
</div>

## 3. Explore the metric view in the low-code UI

With the editor open in **UI** mode, you can inspect each part of the metric view visually before looking at the raw YAML. The UI is in active development, and it's a fast way to understand a metric view someone else authored.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 14px 18px; border-radius: 4px; margin: 12px 0; font-size: 15px; color: #333; line-height: 1.6;">
  <strong style="color: #1565c0; font-size: 16px;">Walk through these in the UI</strong>
  <ul style="margin: 8px 0 0 0; padding-left: 20px;">
    <li><strong>The join:</strong> Find the join to <code style="font-size: 15px;">dim_product</code> and click it. The UI shows the joined table, the join condition, and a link into the <strong>lineage graph</strong>.</li>
    <li><strong>The dimensions:</strong> Click on a dimension such as <strong>Category</strong>. The UI shows the underlying expression and the display name.</li>
    <li><strong>The measures:</strong> Click on a measure such as <strong>Total Revenue</strong>. You will see the expression (<code style="font-size: 15px;">SUM(...)</code>), display name, and any <strong>comments</strong> or <strong>synonyms</strong>. Synonyms let someone ask Genie for "sales" or "turnover" and still hit <code style="font-size: 15px;">total_revenue</code>.</li>
  </ul>
  <p style="margin: 8px 0 0 0;">When done, switch the editor to <strong>YAML</strong> mode (top-right toggle) for the next section.</p>
</div>

## 4. Reading the YAML

A metric view's YAML has several sections. The one you're looking at uses five: `version`, `source`, `joins`, `dimensions`, and `measures`. We'll walk through each.

### 4.1 `version` and `source`

<div style="display: flex; gap: 16px; margin: 12px 0; font-family: -apple-system, sans-serif; align-items: stretch;">
  <div style="flex: 1; background: #f4f6f7; border: 1.5px solid #DCE0E2; border-radius: 8px; padding: 18px; font-family: Consolas, Monaco, monospace; font-size: 17px; color: #1B3139; line-height: 2.2;">
    <span style="color: #1B5162; font-weight: 700;">version:</span> <span style="color: #B26A00;">1.1</span><br/>
    <span style="color: #1B5162; font-weight: 700;">source:</span> &lt;catalog&gt;.vintage_audio.<span style="color: #00A972; font-weight: 700;">fact_orders</span>
  </div>
  <div style="flex: 1.2; display: flex; flex-direction: column; gap: 8px;">
    <div style="background: #f8fafc; border: 2px solid #DCE0E2; border-radius: 8px; padding: 14px 18px; flex: 1; font-size: 16px; display: flex; align-items: center; gap: 12px;">
      <span style="background: #1B5162; color: #fff; padding: 4px 12px; border-radius: 4px; font-weight: 700; font-size: 15px; flex-shrink: 0;">version</span>
      <span style="color: #3d5a65;">Spec format. Keep as <code style="font-size: 16px; background: #EEEDE9; padding: 2px 6px; border-radius: 3px;">1.1</code></span>
    </div>
    <div style="background: #f8fafc; border: 2px solid #DCE0E2; border-radius: 8px; padding: 14px 18px; flex: 1; font-size: 16px; display: flex; align-items: center; gap: 12px;">
      <span style="background: #1B5162; color: #fff; padding: 4px 12px; border-radius: 4px; font-weight: 700; font-size: 15px; flex-shrink: 0;">source</span>
      <span style="color: #3d5a65;">The fact table all measures compute from: <strong>fact_orders</strong></span>
    </div>
  </div>
</div>

### 4.2 `joins`

<div style="display: flex; gap: 16px; margin: 12px 0; font-family: -apple-system, sans-serif; align-items: stretch;">
  <div style="flex: 1; background: #f4f6f7; border: 1.5px solid #DCE0E2; border-radius: 8px; padding: 18px; font-family: Consolas, Monaco, monospace; font-size: 17px; color: #1B3139; line-height: 2;">
    <span style="color: #1B5162; font-weight: 700;">joins:</span><br/>
    &nbsp;&nbsp;- <span style="color: #1B5162; font-weight: 700;">name:</span> <span style="color: #00A972;">dim_product</span><br/>
    &nbsp;&nbsp;&nbsp;&nbsp;<span style="color: #1B5162; font-weight: 700;">source:</span> &lt;catalog&gt;...dim_product<br/>
    &nbsp;&nbsp;&nbsp;&nbsp;<span style="color: #1B5162; font-weight: 700;">"on":</span> <span style="color: #B26A00;">dim_product</span>.product_key = <span style="color: #B26A00;">source</span>.product_key
  </div>
  <div style="flex: 1.2; display: flex; flex-direction: column; gap: 8px;">
    <div style="background: #EDF5F7; border: 2px solid #1B5162; border-radius: 8px; padding: 14px 18px; font-size: 16px; display: flex; align-items: center; gap: 12px;">
      <span style="background: #1B5162; color: #fff; padding: 4px 12px; border-radius: 4px; font-weight: 700; font-size: 15px; flex-shrink: 0;">name</span>
      <span style="color: #3d5a65;">Label for this join: <strong>dim_product</strong></span>
    </div>
    <div style="background: #EDF5F7; border: 2px solid #1B5162; border-radius: 8px; padding: 14px 18px; font-size: 16px; display: flex; align-items: center; gap: 12px;">
      <span style="background: #1B5162; color: #fff; padding: 4px 12px; border-radius: 4px; font-weight: 700; font-size: 15px; flex-shrink: 0;">source</span>
      <span style="color: #3d5a65;">The dimension table's full name</span>
    </div>
    <div style="background: #EDF5F7; border: 2px solid #1B5162; border-radius: 8px; padding: 14px 18px; font-size: 16px; display: flex; align-items: center; gap: 12px;">
      <span style="background: #1B5162; color: #fff; padding: 4px 12px; border-radius: 4px; font-weight: 700; font-size: 15px; flex-shrink: 0;">"on"</span>
      <span style="color: #3d5a65;">Join condition: match <strong>product_key</strong> between tables</span>
    </div>
    <div style="font-size: 15px; color: #5E7077; padding: 4px 0;"><code style="font-size: 15px;">source</code> = fact_orders, <code style="font-size: 15px;">dim_product</code> = the join named above. <code style="font-size: 15px;">"on"</code> is quoted because <code style="font-size: 15px;">on</code> is a YAML reserved word.</div>
  </div>
</div>

### 4.3 `dimensions`

**Dimensions are the columns consumers slice and filter by.** This metric view exposes two, both from `dim_product`:

<div style="display: flex; gap: 16px; margin: 12px 0; font-family: -apple-system, sans-serif; align-items: stretch;">
  <div style="flex: 1; background: #f4f6f7; border: 1.5px solid #DCE0E2; border-radius: 8px; padding: 18px; font-family: Consolas, Monaco, monospace; font-size: 17px; color: #1B3139; line-height: 2;">
    <span style="color: #1B5162; font-weight: 700;">dimensions:</span><br/>
    &nbsp;&nbsp;- <span style="color: #1B5162; font-weight: 700;">name:</span> <span style="color: #00A972;">category</span><br/>
    &nbsp;&nbsp;&nbsp;&nbsp;<span style="color: #1B5162; font-weight: 700;">expr:</span> dim_product.category<br/>
    &nbsp;&nbsp;&nbsp;&nbsp;<span style="color: #1B5162; font-weight: 700;">display_name:</span> <span style="color: #B26A00;">Category</span><br/>
    &nbsp;&nbsp;- <span style="color: #1B5162; font-weight: 700;">name:</span> <span style="color: #00A972;">brand</span><br/>
    &nbsp;&nbsp;&nbsp;&nbsp;<span style="color: #1B5162; font-weight: 700;">expr:</span> dim_product.brand<br/>
    &nbsp;&nbsp;&nbsp;&nbsp;<span style="color: #1B5162; font-weight: 700;">display_name:</span> <span style="color: #B26A00;">Brand</span>
  </div>
  <div style="flex: 1.2; display: flex; flex-direction: column; gap: 8px;">
    <div style="background: #EDF5F7; border: 2px solid #1B5162; border-radius: 8px; padding: 14px 18px; font-size: 16px; display: flex; align-items: center; gap: 12px;">
      <span style="background: #1B5162; color: #fff; padding: 4px 12px; border-radius: 4px; font-weight: 700; font-size: 15px; flex-shrink: 0;">name</span>
      <span style="color: #3d5a65;">How SQL queries reference it: <code style="font-size: 15px;">GROUP BY category</code></span>
    </div>
    <div style="background: #EDF5F7; border: 2px solid #1B5162; border-radius: 8px; padding: 14px 18px; font-size: 16px; display: flex; align-items: center; gap: 12px;">
      <span style="background: #1B5162; color: #fff; padding: 4px 12px; border-radius: 4px; font-weight: 700; font-size: 15px; flex-shrink: 0;">expr</span>
      <span style="color: #3d5a65;">The underlying column or SQL expression</span>
    </div>
    <div style="background: #EDF5F7; border: 2px solid #1B5162; border-radius: 8px; padding: 14px 18px; font-size: 16px; display: flex; align-items: center; gap: 12px;">
      <span style="background: #1B5162; color: #fff; padding: 4px 12px; border-radius: 4px; font-weight: 700; font-size: 15px; flex-shrink: 0;">display_name</span>
      <span style="color: #3d5a65;">Friendly label for dashboards and Genie: <strong>Category</strong></span>
    </div>
  </div>
</div>

### 4.4 `measures`

**Measures are aggregations with built-in logic.** The query writer doesn't choose SUM vs AVG vs COUNT -- the metric view already knows.

<div style="display: flex; gap: 16px; margin: 12px 0; font-family: -apple-system, sans-serif; align-items: stretch;">
  <div style="flex: 1; background: #f4f6f7; border: 1.5px solid #DCE0E2; border-radius: 8px; padding: 18px; font-family: Consolas, Monaco, monospace; font-size: 17px; color: #1B3139; line-height: 2;">
    <span style="color: #1B5162; font-weight: 700;">measures:</span><br/>
    &nbsp;&nbsp;- <span style="color: #1B5162; font-weight: 700;">name:</span> <span style="color: #00A972;">total_revenue</span><br/>
    &nbsp;&nbsp;&nbsp;&nbsp;<span style="color: #1B5162; font-weight: 700;">expr:</span> <span style="color: #00A972;">SUM</span>(source.total_amount)<br/>
    &nbsp;&nbsp;&nbsp;&nbsp;<span style="color: #1B5162; font-weight: 700;">display_name:</span> <span style="color: #B26A00;">Total Revenue</span><br/>
    &nbsp;&nbsp;- <span style="color: #1B5162; font-weight: 700;">name:</span> <span style="color: #00A972;">order_count</span><br/>
    &nbsp;&nbsp;&nbsp;&nbsp;<span style="color: #1B5162; font-weight: 700;">expr:</span> <span style="color: #00A972;">COUNT</span>(source.order_id)<br/>
    &nbsp;&nbsp;&nbsp;&nbsp;<span style="color: #1B5162; font-weight: 700;">display_name:</span> <span style="color: #B26A00;">Order Count</span>
  </div>
  <div style="flex: 1.2; display: flex; flex-direction: column; gap: 8px;">
    <div style="background: #EDF5F7; border: 2px solid #1B5162; border-radius: 8px; padding: 14px 18px; font-size: 16px; display: flex; align-items: center; gap: 12px;">
      <span style="background: #1B5162; color: #fff; padding: 4px 12px; border-radius: 4px; font-weight: 700; font-size: 15px; flex-shrink: 0;">name</span>
      <span style="color: #3d5a65;">How SQL references it: <code style="font-size: 15px;">MEASURE(total_revenue)</code></span>
    </div>
    <div style="background: #EDF5F7; border: 2px solid #1B5162; border-radius: 8px; padding: 14px 18px; font-size: 16px; display: flex; align-items: center; gap: 12px;">
      <span style="background: #1B5162; color: #fff; padding: 4px 12px; border-radius: 4px; font-weight: 700; font-size: 15px; flex-shrink: 0;">expr</span>
      <span style="color: #3d5a65;">The aggregation: <strong>SUM(...)</strong> or <strong>COUNT(...)</strong> built in</span>
    </div>
    <div style="background: #EDF5F7; border: 2px solid #1B5162; border-radius: 8px; padding: 14px 18px; font-size: 16px; display: flex; align-items: center; gap: 12px;">
      <span style="background: #1B5162; color: #fff; padding: 4px 12px; border-radius: 4px; font-weight: 700; font-size: 15px; flex-shrink: 0;">display_name</span>
      <span style="color: #3d5a65;">Friendly label: <strong>Total Revenue</strong>, <strong>Order Count</strong></span>
    </div>
  </div>
</div>

## 5. Putting it all together

Now that you have walked through each section, the diagram below summarizes what you have just read.

<div style="width: 100%; max-width: 1120px; margin: 16px auto;">
<svg viewBox="0 0 920 470" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" style="width: 100%; height: auto; display: block;">
<text x="460" y="30" text-anchor="middle" fill="#1B5162" font-size="19" font-weight="700">ANATOMY OF A METRIC VIEW</text>
<text x="460" y="54" text-anchor="middle" fill="#5E7077" font-size="15" font-style="italic">The YAML you just read, broken down section by section.</text>
<rect x="20" y="78" width="360" height="372" rx="10" fill="#F9F7F4" stroke="#1B5162" stroke-width="1.5"/>
<rect x="40" y="98" width="320" height="56" rx="5" fill="#FFFFFF" stroke="#1B5162" stroke-width="1.4"/>
<text x="54" y="122" fill="#FE3722" font-size="15" font-weight="700">version</text>
<text x="54" y="142" fill="#5E7077" font-size="15" font-style="italic">spec version</text>
<rect x="40" y="162" width="320" height="56" rx="5" fill="#FFFFFF" stroke="#1B5162" stroke-width="1.4"/>
<text x="54" y="186" fill="#FE3722" font-size="15" font-weight="700">source</text>
<text x="54" y="206" fill="#5E7077" font-size="15" font-style="italic">the fact table</text>
<rect x="40" y="226" width="320" height="56" rx="5" fill="#FFFFFF" stroke="#1B5162" stroke-width="1.4"/>
<text x="54" y="250" fill="#FE3722" font-size="15" font-weight="700">joins</text>
<text x="54" y="270" fill="#5E7077" font-size="15" font-style="italic">bring in dimension tables</text>
<rect x="40" y="290" width="320" height="66" rx="5" fill="#FFFFFF" stroke="#1B5162" stroke-width="1.4"/>
<text x="54" y="314" fill="#FE3722" font-size="15" font-weight="700">dimensions</text>
<text x="54" y="334" fill="#5E7077" font-size="15" font-style="italic">slice &amp; filter by &#8212; category, brand</text>
<rect x="40" y="364" width="320" height="70" rx="5" fill="#FFFFFF" stroke="#FE3722" stroke-width="2"/>
<text x="54" y="388" fill="#FE3722" font-size="15" font-weight="700">measures</text>
<text x="54" y="408" fill="#5E7077" font-size="15" font-style="italic">aggregations &#8212; total_revenue,</text>
<text x="54" y="426" fill="#5E7077" font-size="15" font-style="italic">order_count</text>
<rect x="400" y="78" width="500" height="372" rx="8" fill="#FFF7EC" stroke="#FF9800" stroke-width="2"/>
<text x="420" y="106" fill="#B26A00" font-size="15" font-weight="700" letter-spacing="1">HOW TO QUERY IT</text>
<text x="420" y="132" fill="#0b2026" font-size="15" font-weight="700">Dimensions: referenced by name as columns.</text>
<text x="420" y="154" fill="#0b2026" font-size="15" font-weight="700">Measures: wrapped in MEASURE().</text>
<rect x="420" y="172" width="460" height="200" rx="6" fill="#FFFFFF" stroke="#FF9800" stroke-width="1"/>
<text x="436" y="200" fill="#0b2026" font-size="15" font-family="monospace">SELECT</text>
<text x="436" y="222" fill="#0b2026" font-size="15" font-family="monospace">  category,</text>
<text x="436" y="244" fill="#FE3722" font-size="15" font-family="monospace">  MEASURE(total_revenue),</text>
<text x="436" y="266" fill="#FE3722" font-size="15" font-family="monospace">  MEASURE(order_count)</text>
<text x="436" y="288" fill="#0b2026" font-size="15" font-family="monospace">FROM vw_product_metrics</text>
<text x="436" y="310" fill="#0b2026" font-size="15" font-family="monospace">GROUP BY ALL;</text>
<text x="420" y="402" fill="#5E7077" font-size="15" font-style="italic">The same query works from the SQL Editor,</text>
<text x="420" y="422" fill="#5E7077" font-size="15" font-style="italic">dashboards, Genie Agents, and external BI tools.</text>
</svg>
</div>

## 6. Query the metric view

Now that you have explored the metric view in Catalog Explorer and read its YAML, let's query it from this notebook.

Metric views look like regular views, but querying them has two rules and one helpful shorthand:

<div style="width: 100%; margin: 16px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
  <div style="display: flex; gap: 14px;">
    <div style="flex: 1; border: 2px solid #1B5162; border-radius: 10px; overflow: hidden;">
      <div style="background: #1B5162; color: #fff; padding: 10px 14px; font-size: 16px; font-weight: 700; text-align: center;">Rule: No SELECT *</div>
      <div style="padding: 12px 14px; background: #f8fafc; font-size: 15px; color: #3d5a65; line-height: 1.5;">
        <div style="margin-bottom: 6px;">Pick dimensions and measures explicitly.</div>
        <div style="background: #FFF7EC; border: 1px solid #FF9800; border-radius: 4px; padding: 6px 10px; font-size: 14px;"><code style="font-size: 14px;">SELECT *</code> errors because columns are split into dimensions and measures.</div>
      </div>
    </div>
    <div style="flex: 1; border: 2px solid #FE3722; border-radius: 10px; overflow: hidden;">
      <div style="background: #FE3722; color: #fff; padding: 10px 14px; font-size: 16px; font-weight: 700; text-align: center;">Rule: Wrap measures in MEASURE()</div>
      <div style="padding: 12px 14px; background: #FFF8F7; font-size: 15px; color: #3d5a65; line-height: 1.5;">
        <div style="margin-bottom: 6px;"><code style="font-size: 15px;">MEASURE(total_revenue)</code> not <code style="font-size: 15px;">total_revenue</code></div>
        <div style="background: #FFF7EC; border: 1px solid #FF9800; border-radius: 4px; padding: 6px 10px; font-size: 14px;">Bare measure names error. Measures carry built-in aggregation logic.</div>
      </div>
    </div>
    <div style="flex: 1; border: 2px solid #00A972; border-radius: 10px; overflow: hidden;">
      <div style="background: #00A972; color: #fff; padding: 10px 14px; font-size: 16px; font-weight: 700; text-align: center;">Tip: GROUP BY ALL</div>
      <div style="padding: 12px 14px; background: #F0FAF5; font-size: 15px; color: #3d5a65; line-height: 1.5;">
        <div style="margin-bottom: 6px;">When your SELECT includes dimensions, you need a GROUP BY. <code style="font-size: 15px;">GROUP BY ALL</code> is the shorthand.</div>
        <div style="font-size: 14px; color: #5E7077;">If you only select measures (no dimensions), skip GROUP BY entirely to get grand totals.</div>
      </div>
    </div>
  </div>
</div>

The first query uses all three. Run it.

In [0]:
%sql
SELECT
  category,
  MEASURE(total_revenue) AS revenue,
  MEASURE(order_count)   AS orders
FROM vw_product_metrics
GROUP BY ALL
ORDER BY revenue DESC;

You should see five rows, one per product category, sorted by revenue. The `MEASURE()` function evaluates each measure's stored aggregation against the current grouping. `GROUP BY ALL` tells Databricks to group by every non-measure column in the `SELECT` list, which saves you from listing them again.

### One more query: aggregate across everything

If you omit the dimension and just select measures, you get one row with the totals:

In [0]:
%sql
SELECT
  MEASURE(total_revenue) AS total_revenue,
  MEASURE(order_count)   AS order_count
FROM vw_product_metrics;

One row, two totals. Same metric view, different question, same definitions of `total_revenue` and `order_count`.

### Use Genie Code to explain a query

Genie Code isn't just for writing SQL, it's good at *explaining* it. The inline box can't read an existing query, so use the **sidebar** for this one.

<div style="width: 100%; margin: 12px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; font-size: 16px;">
  <div style="display: flex; gap: 0; align-items: stretch;">
    <div style="flex: 1; border: 2px solid #1B5162; border-radius: 10px; overflow: hidden; display: flex; flex-direction: column;">
      <div style="background: #1B5162; color: #fff; padding: 10px 14px; font-size: 16px; font-weight: 700; display: flex; align-items: center; gap: 8px;">
        <span style="background: rgba(255,255,255,0.2); border-radius: 50%; width: 26px; height: 26px; display: inline-flex; align-items: center; justify-content: center; font-size: 15px; flex-shrink: 0;">1</span> Select the query cell
      </div>
      <div style="padding: 14px; background: #f8fafc; flex: 1; color: #3d5a65; line-height: 1.6;">Click the all-totals query cell above to select it</div>
    </div>
    <div style="flex: 0 0 40px; display: flex; align-items: center; justify-content: center; font-size: 24px; color: #1B5162;">&#x2192;</div>
    <div style="flex: 1; border: 2px solid #1B5162; border-radius: 10px; overflow: hidden; display: flex; flex-direction: column;">
      <div style="background: #1B5162; color: #fff; padding: 10px 14px; font-size: 16px; font-weight: 700; display: flex; align-items: center; gap: 8px;">
        <span style="background: rgba(255,255,255,0.2); border-radius: 50%; width: 26px; height: 26px; display: inline-flex; align-items: center; justify-content: center; font-size: 15px; flex-shrink: 0;">2</span> Open sidebar Genie
      </div>
      <div style="padding: 14px; background: #f8fafc; flex: 1; color: #3d5a65; line-height: 1.6;">Click the <strong>Genie / Assistant</strong> icon on the right edge of the workspace</div>
    </div>
    <div style="flex: 0 0 40px; display: flex; align-items: center; justify-content: center; font-size: 24px; color: #1B5162;">&#x2192;</div>
    <div style="flex: 1.4; border: 2px solid #1B5162; border-radius: 10px; overflow: hidden; display: flex; flex-direction: column;">
      <div style="background: #1B5162; color: #fff; padding: 10px 14px; font-size: 16px; font-weight: 700; display: flex; align-items: center; gap: 8px;">
        <span style="background: rgba(255,255,255,0.2); border-radius: 50%; width: 26px; height: 26px; display: inline-flex; align-items: center; justify-content: center; font-size: 15px; flex-shrink: 0;">3</span> Ask this
      </div>
      <div style="padding: 14px; background: #f8fafc; flex: 1;">
        <div id="genie-prompt-10" style="background: #fff; border: 1.5px solid #DCE0E2; border-radius: 8px; padding: 12px 14px; color: #3d5a65; line-height: 1.5; font-style: italic;">Explain what this query does and why the measures are wrapped in MEASURE().</div>
        <button onclick="var t=document.getElementById('genie-prompt-10').textContent,a=document.createElement('textarea');a.value=t;a.style.position='fixed';a.style.left='-9999px';document.body.appendChild(a);a.select();document.execCommand('copy');document.body.removeChild(a);this.textContent='Copied!';this.style.background='#00A972';this.style.borderColor='#00A972';var b=this;setTimeout(function(){b.textContent='Copy prompt';b.style.background='#1B5162';b.style.borderColor='#1B5162'},1500)" style="margin-top: 10px; background: #1B5162; color: #fff; border: 2px solid #1B5162; border-radius: 6px; padding: 6px 18px; font-size: 16px; font-weight: 600; cursor: pointer;">Copy prompt</button>
      </div>
    </div>
  </div>
</div>

Genie should explain that `MEASURE()` evaluates each measure's stored aggregation from the metric view definition, and that without a `GROUP BY`, the measures aggregate across all rows to produce grand totals. Asking AI to explain unfamiliar syntax is a fast way to build understanding when you're new to a feature.

## 7. The key takeaway: one definition, changed in one place

<div style="border-left: 5px solid #FE3722; background: #FDEBE8; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<strong style="display:block; color:#C62828; font-size:1.08em; margin-bottom:8px;">This is the whole point of a metric view</strong>
<div style="color:#333;">
The measure <code>total_revenue</code> is defined <strong>once</strong>, as <code>SUM(source.total_amount)</code>, inside the metric view's YAML. Every consumer: the SQL Editor, dashboards, Genie Agents, alerts, Power BI evaluates that same definition. There is no second place where "revenue" is defined, so two teams can't end up with two different revenue numbers.
<br><br>
If the business later redefines revenue (say, revenue should be net of tax), you change the <code>expr:</code> in <strong>one place</strong> and every query, dashboard, and report picks it up automatically. You'll do exactly that in the next step.
</div>
</div>

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 14px 18px; border-radius: 4px; margin: 12px 0; font-size: 15px; color: #333; line-height: 1.6;">
  <strong style="color: #1565c0; font-size: 16px;">Try it (optional)</strong>
  <ol style="margin: 8px 0 0 0; padding-left: 20px;">
    <li>In the Catalog Explorer editor, change the <code style="font-size: 15px;">total_revenue</code> measure's <code style="font-size: 15px;">expr:</code> from <code style="font-size: 15px;">SUM(source.total_amount)</code> to <code style="font-size: 15px;">ROUND(SUM(source.total_amount), 0)</code> and <strong>Save</strong>.</li>
    <li>Re-run the category query above. The revenue values update with <strong>no change to the query itself</strong>.</li>
    <li>Change it back afterward if you like.</li>
  </ol>
  <p style="margin: 8px 0 0 0;">This is what "single source of truth" means in practice.</p>
</div>

## Going further (optional)

You now know how to read, query, and author a metric view &mdash; that's the core. Two capabilities exist for when your metrics grow beyond this course. Neither is needed here; they're just worth knowing about.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 14px 18px; border-radius: 4px; margin: 12px 0; font-size: 15px; color: #333; line-height: 1.6;">
  <ul style="margin: 4px 0 0 0; padding-left: 20px;">
    <li><strong>Materialization</strong> &mdash; a metric view can pre-compute its aggregations so heavy queries return faster. You add a <code style="font-size: 14px;">materialization</code> section to the YAML, and Databricks keeps the results refreshed and automatically chooses between the pre-computed data and the source table for each query. <em>Generally available.</em></li>
    <li><strong>Sharing with OpenSharing</strong> &mdash; you can share a metric view with Databricks users in other metastores or accounts, so the same trusted measure definitions travel beyond your own workspace. <em>Beta.</em></li>
  </ul>
  <p style="margin: 8px 0 0 0;">Both are out of scope for this course. See the links in <strong>Summary and Next Steps</strong> if you want to read more later.</p>
</div>

## What you did in this lesson

<div style="border-left: 4px solid #4caf50; background: #e8f5e9; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
<div style="color:#333;">
You learned how to read and query a metric view:

| Step | What you did |
|---|---|
| Understood why metric views matter | One definition, many consumers (SQL Editor, dashboards, Genie Agents, alerts, external BI), no duplicate metric logic |
| Opened the metric view in Catalog Explorer | Toured the Overview, Details, and Lineage tabs |
| Explored the low-code UI | Clicked the join (lineage context), measures (expr, comments, synonyms), and dimensions |
| Walked through the YAML | <code>version</code>, <code>source</code>, <code>joins</code>, <code>dimensions</code>, <code>measures</code> |
| Queried the metric view | Three rules: no <code>SELECT *</code>, wrap measures in <code>MEASURE()</code>, use <code>GROUP BY ALL</code> |
| Used Genie Code | To explain a metric-view query |

</div>
</div>


&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>